# YOLOv8n Improved (Augmentations, Oversample)

Pipeline:
1. Làm sạch
2. COCO sang YOLO
3. Chia tập
4. Letterbox (giữ Aspect Ratio)
5. Oversample (Kèm Offline Data Augmentation)
6. Huấn luyện (áp dụng Augmentation của YOLO & TTA)

In [1]:
!git clone https://github.com/Shiba-dotcom/waste-detection_project.git

Cloning into 'waste-detection_project'...
remote: Enumerating objects: 3498, done.
remote: Counting objects: 100% (95/95), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 3498 (delta 39), reused 55 (delta 24), pack-reused 3403 (from 2)
Receiving objects: 100% (3498/3498), 2.53 GiB | 52.63 MiB/s, done.
Resolving deltas: 100% (156/156), done.


In [2]:
!pip install gdown
# Tải file từ Drive về Kaggle
!gdown --id "17Nmi2fonq31N1PZUlCZ0q7FsXf2JzGqM" -O raw.zip

# Chỉ cần tạo đến thư mục data (thư mục raw sẽ tự sinh ra khi giải nén)
!mkdir -p /kaggle/working/waste-detection_project/data

# Giải nén vào thư mục data
!unzip -q raw.zip -d /kaggle/working/waste-detection_project/data/

# Dọn dẹp file zip để tránh bị đầy bộ nhớ (Disk quota exceeded)
!rm raw.zip

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=17Nmi2fonq31N1PZUlCZ0q7FsXf2JzGqM
From (redirected): https://drive.google.com/uc?id=17Nmi2fonq31N1PZUlCZ0q7FsXf2JzGqM&confirm=t&uuid=7ba8aaa5-61bb-4708-a37a-fd0cfb17e90f
To: /kaggle/working/raw.zip
100%|███████████████████████████████████████| 2.62G/2.62G [00:25<00:00, 103MB/s]


In [3]:
# Di chuyển vào đúng thư mục của script rồi mới chạy
%cd /kaggle/working/waste-detection_project/src/data_prep
!python data_cleaning.py

%cd /kaggle/working/waste-detection_project/src
!python Training_dataYolo.py

%cd /kaggle/working/waste-detection_project/src/data_prep
!python split_dataset.py

/kaggle/working/waste-detection_project/src/data_prep
  DATA CLEANING - TACO Dataset

[Load] annotations.json ...
  So anh goc         : 1500
  So annotations goc : 4784
  So categories      : 60

────────────────────────────────────────────────────────────
[Buoc 1] Kiem tra dong trung lap (Duplicates)
────────────────────────────────────────────────────────────
  Duplicate annotations: 0
  Duplicate image IDs: 0
  Duplicate file names: 0

────────────────────────────────────────────────────────────
[Buoc 2] Kiem tra gia tri thieu (Missing Values)
────────────────────────────────────────────────────────────
  Images: Tat ca truong bat buoc day du
  Annotations: Tat ca truong bat buoc day du
  Anh khong co annotation: 0

────────────────────────────────────────────────────────────
[Buoc 3] Kiem tra nhan khong hop le
────────────────────────────────────────────────────────────
  Annotations voi category_id khong hop le: 0
  Categories khong co trong mapping.csv: 1
    - 'Rope & strings' 

In [4]:
# Tiếp tục đứng ở data_prep để chạy các kĩ thuật nâng cao
%cd /kaggle/working/waste-detection_project/src/data_prep
!python letterbox.py
!python oversample.py

# QUAN TRỌNG: Xong xuôi hết thì dời máy ảo quay lại đúng thư mục experimental để hàm model.train() chạy trơn tru
%cd /kaggle/working/waste-detection_project/notebooks/experimental

/kaggle/working/waste-detection_project/src/data_prep
[INFO] Bat dau ap dung Letterbox Padding (640x640) cho toan bo dataset...
[INFO] Hoan thanh! Da xu ly letterbox cho 833 anh va cap nhat file nhãn (.txt) tuong ung.
[INFO] Bat dau Oversampling voi Data Augmentation...
Oversampled Glass images: 741 (x2 = 1482 copies)
Oversampled Paper images: 242 (x1 = 242 copies)
Oversampled Metal images: 393 (x1 = 393 copies)
[INFO] Done.
/kaggle/working/waste-detection_project/notebooks/experimental


In [8]:
%cd /kaggle/working/waste-detection_project
!pip install -r requirements.txt
%cd /kaggle/working/waste-detection_project/notebooks/experimental

/kaggle/working/waste-detection_project
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.8 MB/s eta 0:00:00a 0:00:01
/kaggle/working/waste-detection_project/notebooks/experimental


In [11]:
yaml_file = '/kaggle/working/waste-detection_project/data/processed/dataset.yaml'

with open(yaml_file, 'r') as f:
    content = f.read()

# Sửa đường dẫn ảo thành đường dẫn tuyệt đối của Kaggle
content = content.replace('path: ../data/processed', 'path: /kaggle/working/waste-detection_project/data/processed')

with open(yaml_file, 'w') as f:
    f.write(content)

print("Đã sửa đường dẫn file dataset.yaml thành công!")

Đã sửa đường dẫn file dataset.yaml thành công!


In [12]:
import os, time, glob
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

BASE_DIR = Path.cwd().parent.parent
YAML_PATH = BASE_DIR / "data/processed/dataset.yaml"
RESULTS = BASE_DIR / "results"
RESULTS.mkdir(exist_ok=True)

model_path = BASE_DIR / "models/yolov8n.pt"
model = YOLO(str(model_path))
# Train YOLOv8n with strong augmentations
train_results = model.train(
    data = str(YAML_PATH),
    epochs = 50,
    imgsz = 640,
    batch = 16,
    name = "yolov8n_improved",
    project = str(RESULTS / "runs"),
    exist_ok = True,
    verbose = True,
    # Augmentations enabled
    degrees = 10.0,
    translate = 0.1,
    scale = 0.5,
    fliplr = 0.5,
    hsv_s = 0.7,
    hsv_v = 0.4,
    mosaic = 1.0,
    mixup = 0.1
)
# Evaluation with TTA
print("\n[2] Danh gia tren tap Test (TTA) ...")
metrics = model.val(split="test", augment=True, verbose=False)

map50 = float(metrics.box.map50)
print(f"\nTTA mAP@0.5: {map50:.4f}")

Ultralytics 8.4.52 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/waste-detection_project/data/processed/dataset.yaml, degrees=10.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=/kaggle/working/waste-detection_project/models/yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_improved,

In [13]:
!zip -r yolov8n_improved_results.zip /kaggle/working/waste-detection_project/results/runs/yolov8n_improved

  adding: kaggle/working/waste-detection_project/results/runs/yolov8n_improved/ (stored 0%)
  adding: kaggle/working/waste-detection_project/results/runs/yolov8n_improved/args.yaml (deflated 56%)
  adding: kaggle/working/waste-detection_project/results/runs/yolov8n_improved/BoxP_curve.png (deflated 7%)
  adding: kaggle/working/waste-detection_project/results/runs/yolov8n_improved/train_batch2.jpg (deflated 2%)
  adding: kaggle/working/waste-detection_project/results/runs/yolov8n_improved/BoxF1_curve.png (deflated 11%)
  adding: kaggle/working/waste-detection_project/results/runs/yolov8n_improved/train_batch7920.jpg (deflated 8%)
  adding: kaggle/working/waste-detection_project/results/runs/yolov8n_improved/BoxPR_curve.png (deflated 13%)
  adding: kaggle/working/waste-detection_project/results/runs/yolov8n_improved/BoxR_curve.png (deflated 12%)
  adding: kaggle/working/waste-detection_project/results/runs/yolov8n_improved/val_batch1_pred.jpg (deflated 7%)
  adding: kaggle/working/waste-